# Phase 9 Monitoring Simulation

Runs the two monitoring instruments against real vintages and shows why both are needed.

The split is forced by the data, and it mirrors production exactly. **2017-2018 has no
labels** - those vintages cannot reach 24 months of observation before the 2018-12 cutoff -
so only distribution drift can be measured on them. That is precisely the situation a live
monitoring job is in: it sees applications long before it sees outcomes.

**2015-2016 has labels**, so the outcome check can be backtested there. That is the
convincing test, because the answer is already known: PSI came back stable while the default
rate rose 27% relative.

Note there is no `build_target` call for the drift sections. Monitoring scores applications
whose outcome does not exist yet, and the pipeline should not pretend otherwise.


In [1]:
from pathlib import Path

import numpy as np
import polars as pl

from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.target import build_target
from credit_risk.features.build_dataset import assemble_feature_matrix
from credit_risk.features.cleaning import clean_features
from credit_risk.monitoring.drift import DriftMonitor
from credit_risk.monitoring.performance import (
    early_warning,
    expected_vs_actual,
    vintage_performance,
)
from credit_risk.serving.artifacts import ChampionBundle

pl.Config.set_tbl_rows(40)
CONFIG_PATH = Path("../configs/base.yaml")
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")

bundle = ChampionBundle.load(Path("../artifacts/champion"))
monitor = DriftMonitor(bundle)
print("model", bundle.metadata["input_sha256"][:12], "|", len(bundle.features), "features")
print("reference profiles:", len(bundle.reference["features"]), "numeric features + score")


model 3eae03c28fd9 | 69 features
reference profiles: 58 numeric features + score


In [2]:
raw = load_raw_accepted_loans(DATA_PATH)

# No build_target: these vintages have no outcome, which is the point.
cleaned = clean_features(raw).with_columns(
    pl.col("issue_d").cast(pl.Utf8).str.strptime(pl.Date, "%b-%Y", strict=False).alias("_issued")
)
unlabelled = {
    year: cleaned.filter(pl.col("_issued").dt.year() == year) for year in (2017, 2018)
}
print({y: df.height for y, df in unlabelled.items()})


{2017: 443579, 2018: 495242}


## 1. Score drift on vintages that have no outcome yet

The single most informative number a monitoring job produces. If the score distribution has
moved, the population the model is being asked about is no longer the one it was fitted on.

Bands: < 0.10 stable, 0.10-0.25 watch, > 0.25 material.


In [3]:
for year, df in unlabelled.items():
    scores = bundle.predict_score(df)
    result = monitor.score_drift(scores)
    print(f"{year}  n={df.height:>7}  PSI={result['psi']:.4f}  "
          f"out_of_range={result['out_of_range']:.3f}  {result['band']:<18} "
          f"median score {np.median(scores):.0f}  mean PD {bundle.predict_pd(df).mean():.4f}")

# out_of_range near zero confirms the reference and the live data are the same quantity.
# A large value means a configuration bug, not drift - PSI is meaningless in that case.


2017  n= 443579  PSI=0.0111  out_of_range=0.010  stable             median score 557  mean PD 0.0969
2018  n= 495242  PSI=0.0292  out_of_range=0.026  stable             median score 559  mean PD 0.0940


In [4]:
# Per quarter, because an annual figure averages away a shift that started mid-year.
quarterly = (
    cleaned.filter(pl.col("_issued").dt.year().is_in([2017, 2018]))
    .with_columns(
        (pl.col("_issued").dt.year().cast(pl.Utf8) + "Q"
         + ((pl.col("_issued").dt.month() - 1) // 3 + 1).cast(pl.Utf8)).alias("quarter")
    )
)
rows = []
for quarter in sorted(quarterly["quarter"].unique().to_list()):
    part = quarterly.filter(pl.col("quarter") == quarter)
    scores = bundle.predict_score(part)
    rows.append({"quarter": quarter, "n": part.height,
                 "median_score": float(np.median(scores)),
                 "mean_pd": float(bundle.predict_pd(part).mean()),
                 **{k: v for k, v in monitor.score_drift(scores).items() if k != "metric"}})
pl.DataFrame(rows)


quarter,n,median_score,mean_pd,psi,out_of_range,band
str,i64,f64,f64,f64,f64,str
"""2017Q1""",96779,557.737364,0.094544,0.0169,0.0,"""stable"""
"""2017Q2""",105451,557.128673,0.096308,0.0116,0.003,"""stable"""
"""2017Q3""",122701,556.311402,0.099777,0.0049,0.0132,"""stable"""
"""2017Q4""",118648,558.183615,0.09631,0.0183,0.0223,"""stable"""
"""2018Q1""",107864,558.632678,0.096003,0.0272,0.0372,"""stable"""
"""2018Q2""",130772,557.950557,0.097848,0.0205,0.0362,"""stable"""
"""2018Q3""",128194,559.910002,0.091773,0.0395,0.0228,"""stable"""
"""2018Q4""",128412,559.868077,0.090803,0.0399,0.0077,"""stable"""


## 2. Feature drift — localising a shift, not just detecting one

Score PSI says something moved. This says what. A column reported as `column missing` is the
more dangerous finding: the model scores an absent column as null without complaining, so
every score shifts and nothing fails.


In [5]:
for year, df in unlabelled.items():
    report = monitor.report(df, bundle.predict_score(df))
    print(f"=== {year} ===  score PSI {report['score']['psi']} ({report['score']['band']})")
    print(f"  {report['n_features_flagged']}/{report['n_features_checked']} features flagged")
    for entry in report["flagged"][:10]:
        print(f"    {entry['feature']:<30} psi={entry['psi']} {entry['band']}")
    print()


=== 2017 ===  score PSI 0.0111 (stable)
  5/58 features flagged
    percent_bc_gt_75               psi=0.1711 watch
    bc_util                        psi=0.1523 watch
    revol_util                     psi=0.1512 watch
    num_bc_tl                      psi=0.1236 watch
    bc_open_to_buy                 psi=0.1171 watch

=== 2018 ===  score PSI 0.0292 (stable)
  8/58 features flagged
    percent_bc_gt_75               psi=0.3359 material shift
    bc_util                        psi=0.3084 material shift
    revol_util                     psi=0.296 material shift
    bc_open_to_buy                 psi=0.2541 material shift
    fico_range_low                 psi=0.1605 watch
    num_bc_tl                      psi=0.1527 watch
    inq_last_6mths                 psi=0.1316 watch
    num_rev_accts                  psi=0.1262 watch



In [6]:
drift_2018 = monitor.feature_drift(unlabelled[2018])
drift_2018.write_csv("../docs/monitoring_feature_drift_2018.csv")
drift_2018.head(20)


feature,psi,out_of_range,band
str,f64,f64,str
"""percent_bc_gt_75""",0.3359,0.2263,"""material shift"""
"""bc_util""",0.3084,0.1112,"""material shift"""
"""revol_util""",0.296,0.1209,"""material shift"""
"""bc_open_to_buy""",0.2541,0.0677,"""material shift"""
"""fico_range_low""",0.1605,0.0558,"""watch"""
"""num_bc_tl""",0.1527,0.0634,"""watch"""
"""inq_last_6mths""",0.1316,0.0922,"""watch"""
"""num_rev_accts""",0.1262,0.0967,"""watch"""
"""total_acc""",0.0952,0.0703,"""stable"""


## 3. Backtest: the check PSI cannot perform

2015 and 2016 have labels, so both instruments can be run side by side on the same data.
This is the whole argument for Phase 9 in one table.

Expected outcome, from Phase 5: distributions stable, outcomes materially worse in 2016.
If the drift row is green and the performance row is red, the two instruments are measuring
different things and neither substitutes for the other.


In [7]:
labelled = assemble_feature_matrix(build_target(raw, CONFIG_PATH), CONFIG_PATH)

# Two PDs, because they answer different questions. `pd` carries the central tendency
# anchor, so it is deliberately not the point-in-time rate of any single vintage - a
# steady gap against one cohort is the anchor working. `pd_pit` drops the anchor and is
# the right target for "was this vintage as predicted".
raw_scores = bundle.predict_raw(labelled)
clipped = np.clip(raw_scores, 1e-9, 1 - 1e-9)
logits = np.log(clipped / (1 - clipped))
cal = bundle.calibrator
pd_pit = 1 / (1 + np.exp(-(cal["coef"] * logits + cal["intercept"])))
pd_pit = np.clip(pd_pit, cal["floor"], cal["cap"])

scored = labelled.with_columns(
    pl.Series("pd", bundle.predict_pd(labelled)),
    pl.Series("pd_pit", pd_pit),
    pl.col("issue_d").cast(pl.Utf8).str.strptime(pl.Date, "%b-%Y", strict=False)
      .dt.year().cast(pl.Utf8).alias("vintage"),
)
print({s: scored.filter(pl.col("split") == s).height for s in ("train", "validation", "oot_test")})
print("mean anchored PD", round(float(scored['pd'].mean()), 4),
      "| mean point-in-time PD", round(float(scored['pd_pit'].mean()), 4))


{'train': 370443, 'validation': 421095, 'oot_test': 434407}
mean anchored PD 0.1051 | mean point-in-time PD 0.1036


In [8]:
# Same table on the un-anchored PD. 2013-2014 should move much closer to a ratio of 1:
# their apparent deterioration under the anchored PD is the long-run adjustment, not drift.
print("--- outcome performance, point-in-time PD ---")
print(vintage_performance(scored, "pd_pit", "vintage").to_pandas().to_string(index=False))


--- outcome performance, point-in-time PD ---
vintage      n  expected_defaults  actual_defaults  expected_rate  actual_rate    ratio  z_score  significant ratio_band  alert
   2013 134814            13107.2            11220       0.097224     0.083226 0.856021   -17.79         True   material   True
   2014 235629            24593.7            21835       0.104375     0.092667 0.887830   -19.09         True   material   True
   2015 421095            45052.5            45051       0.106989     0.106985 0.999967    -0.01        False     stable  False
   2016 434407            44215.4            49327       0.101783     0.113550 1.115607    26.35         True   material   True


## 4. Early warning — a verdict before the window closes

A 24-month horizon means a vintage's real answer arrives two years late. The hazard curve,
measured on the fully matured 2013 vintage, says what share of eventual defaults should have
landed by each month on book, so a young cohort can be tested against a scaled expectation.

The comparison against `coverage = 1.0` is the point: without scaling, a perfectly healthy
cohort looks catastrophically better than predicted, and the alert is meaningless.


In [9]:
# Cumulative share of defaults arriving by each month on book, WITHIN the 24-month
# horizon - not of lifetime defaults, which is the different quantity business.py uses.
matured = scored.filter((pl.col("vintage") == "2013") & (pl.col("default_flag") == 1))
by_mob = (
    matured.group_by(pl.col("mob_event").cast(pl.Int32))
    .agg(pl.len().alias("n"))
    .sort("mob_event")
    .with_columns((pl.col("n").cum_sum() / pl.col("n").sum()).alias("cum_share"))
)
hazard_coverage = dict(
    zip(by_mob["mob_event"].to_list(), by_mob["cum_share"].to_list(), strict=True)
)
for mob in (6, 12, 18, 24):
    print(f"  MOB {mob:>2}: {hazard_coverage.get(mob, float('nan')):.3f} of in-horizon defaults")


  MOB  6: 0.023 of in-horizon defaults
  MOB 12: 0.260 of in-horizon defaults
  MOB 18: 0.640 of in-horizon defaults
  MOB 24: 1.000 of in-horizon defaults


In [10]:
oot = scored.filter(pl.col("split") == "oot_test")

print("2016 vintage, judged early (alert requires both significant AND material):")
for mob in (6, 12, 18, 24):
    r = early_warning(oot, "pd", "mob_event", mob, hazard_coverage).row(0, named=True)
    print(f"  MOB {mob:>2}  coverage={r['coverage']:.3f}  "
          f"expected={r['expected_defaults']:>8.0f}  actual={r['actual_defaults']:>7}  "
          f"ratio={r['ratio']:.3f}  z={r['z_score']:>6.1f}  "
          f"{'ALERT' if r['alert'] else 'ok'}")

# The number that justifies the phase: the earliest MOB at which 2016 already alerts.
earliest = next(
    (m for m in sorted(hazard_coverage)
     if early_warning(oot, "pd", "mob_event", m, hazard_coverage).row(0, named=True)["alert"]),
    None,
)
print(f"\nearliest alerting MOB: {earliest}  "
      f"({24 - earliest} months before the outcome window closes)"
      if earliest else "\n2016 never alerts")


2016 vintage, judged early (alert requires both significant AND material):
  MOB  6  coverage=0.023  expected=    1035  actual=   1171  ratio=1.131  z=   4.2  ALERT
  MOB 12  coverage=0.260  expected=   11674  actual=  14265  ratio=1.222  z=  24.5  ALERT
  MOB 18  coverage=0.640  expected=   28716  actual=  33051  ratio=1.151  z=  26.9  ALERT
  MOB 24  coverage=1.000  expected=   44855  actual=  49327  ratio=1.100  z=  22.9  ALERT

earliest alerting MOB: 5  (19 months before the outcome window closes)


## 5. What a production job would run

Daily or weekly on the applications scored since the last run, plus monthly on maturing
cohorts. The two cadences are different because the two questions are different: drift is
answerable immediately, outcomes are not.


In [11]:
def monitoring_cycle(new_applications: pl.DataFrame, maturing: pl.DataFrame | None = None) -> dict:
    """One monitoring pass: drift on fresh applications, outcomes on cohorts old enough to judge."""
    scores = bundle.predict_score(new_applications)
    result = {"model_version": bundle.metadata["input_sha256"][:12],
              "n_scored": new_applications.height,
              "drift": monitor.report(new_applications, scores)}
    if maturing is not None and maturing.height:
        result["performance"] = expected_vs_actual(
            maturing["pd"].to_numpy(), maturing["default_flag"].to_numpy()
        )
    return result

cycle = monitoring_cycle(unlabelled[2018], maturing=oot)
print("drift      :", cycle["drift"]["score"], f"| {cycle['drift']['n_features_flagged']} features flagged")
print("performance:", cycle["performance"])


drift      : {'metric': 'score', 'psi': 0.0292, 'out_of_range': 0.0256, 'band': 'stable'} | 8 features flagged
performance: {'n': 434407, 'expected_defaults': 44854.9, 'actual_defaults': 49327, 'expected_rate': 0.103255519058102, 'actual_rate': 0.11355019601433679, 'ratio': 1.099700984994729, 'z_score': 22.92, 'significant': True, 'ratio_band': 'watch', 'alert': True}
